# Train MobileNetV2 ? Brain MRI Classification

This notebook trains MobileNetV2 and saves the best checkpoint. **Training only happens when you run cells in this notebook.**

**Prerequisites**
- Run the data-preparation notebook first.
- It must have created `merged_../prepared_data/train.csv`, `merged_../prepared_data/validation.csv`, and `merged_../prepared_data/test.csv`.
- The CSVs use the same fixed splits as the EfficientNetV2-S notebooks.


## 1. Imports and configuration


In [4]:
# %pip install pandas pillow matplotlib scikit-learn torch torchvision
from pathlib import Path
import copy, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DATA_DIR=Path("..")/("merged_prepared_data"); MODEL_DIR=Path("..")/("trained_models"); MODEL_DIR.mkdir(exist_ok=True)
TRAIN_CSV=DATA_DIR/"train.csv"; VAL_CSV=DATA_DIR/"validation.csv"; TEST_CSV=DATA_DIR/"test.csv"
CHECKPOINT_PATH=MODEL_DIR/"mobilenet_v2_merged_best.pth"; HISTORY_PATH=MODEL_DIR/"mobilenet_v2_merged_training_history.csv"
PREDICTIONS_PATH=MODEL_DIR/"mobilenet_v2_merged_test_predictions.csv"; COMPARISON_PATH=MODEL_DIR/"model_comparison.csv"
BATCH_SIZE=32; HEAD_EPOCHS=5; FINETUNE_EPOCHS=20; PATIENCE=5; NUM_WORKERS=0
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP=device.type == "cuda"
print("GPU:",torch.cuda.get_device_name(0) if USE_AMP else "CPU")
print("Mixed precision:",USE_AMP)
print("Device:",device)


GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Mixed precision: True
Device: cuda


## 2. Load the saved splits

This notebook does not recreate or change the split. It uses the CSVs saved during data preparation.


In [5]:
for csv_path in [TRAIN_CSV,VAL_CSV,TEST_CSV]:
    if not csv_path.exists(): raise FileNotFoundError(f"Missing {csv_path}. Run data preparation first.")
train_df=pd.read_csv(TRAIN_CSV); val_df=pd.read_csv(VAL_CSV); test_df=pd.read_csv(TEST_CSV)
required={"filepath","class"}
for name,frame in [("train",train_df),("validation",val_df),("test",test_df)]:
    missing=required-set(frame.columns)
    if missing: raise ValueError(f"{name}.csv is missing columns: {sorted(missing)}")
class_names=sorted(train_df["class"].unique().tolist())
class_to_idx={name:i for i,name in enumerate(class_names)}; idx_to_class={i:name for name,i in class_to_idx.items()}
for name,frame in [("validation",val_df),("test",test_df)]:
    unknown=set(frame["class"].unique())-set(class_names)
    if unknown: raise ValueError(f"Unexpected classes in {name}: {sorted(unknown)}")
print("Classes:",class_to_idx); print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
display(pd.DataFrame({"train":train_df["class"].value_counts(),"validation":val_df["class"].value_counts(),"test":test_df["class"].value_counts()}).fillna(0).astype(int).sort_index())


Classes: {'glioma_tumor': 0, 'meningioma_tumor': 1, 'normal': 2, 'pituitary_tumor': 3}
Train: 6800 | Validation: 1457 | Test: 1458


,train,validation,test
class,,,
glioma_tumor,1865,399,400
meningioma_tumor,1749,375,375
normal,1372,294,294
pituitary_tumor,1814,389,389


## 3. Create datasets and dataloaders

MobileNetV2 uses its standard 224 ? 224 pretrained-input transform.


In [6]:
weights=MobileNet_V2_Weights.DEFAULT
train_transform=transforms.Compose([transforms.Resize((224,224)),transforms.RandomRotation(5),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
eval_transform=weights.transforms()
class BrainMRIDataset(Dataset):
    def __init__(self,dataframe,transform=None): self.dataframe=dataframe.reset_index(drop=True).copy(); self.transform=transform
    def __len__(self): return len(self.dataframe)
    def __getitem__(self,index):
        row=self.dataframe.iloc[index]
        with Image.open(row["filepath"]) as img: image=img.convert("RGB")
        if self.transform: image=self.transform(image)
        return image,class_to_idx[row["class"]]
train_dataset=BrainMRIDataset(train_df,train_transform); val_dataset=BrainMRIDataset(val_df,eval_transform); test_dataset=BrainMRIDataset(test_df,eval_transform)
generator=torch.Generator().manual_seed(SEED)
loader_kwargs={"num_workers": NUM_WORKERS, "pin_memory": USE_AMP}
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=generator,**loader_kwargs)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,**loader_kwargs)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,**loader_kwargs)
print("Training batch shape:",tuple(next(iter(train_loader))[0].shape))


Training batch shape: (32, 3, 224, 224)


## 4. Build and train MobileNetV2

The classifier is trained first with the feature extractor frozen, followed by full-network fine-tuning. The best validation-loss checkpoint is saved with its class metadata.


In [7]:
model=mobilenet_v2(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model=model.to(device)
criterion=nn.CrossEntropyLoss()
scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)

history=[]
best_val_loss=float("inf")
best_model_weights=None
PATIENCE=5
HEAD_EPOCHS=5
FINETUNE_EPOCHS=20

def run_training_phase(optimizer,max_epochs,phase_name):
    global best_val_loss,best_model_weights
    phase_best_val_loss=float("inf")
    epochs_without_improvement=0
    for epoch in range(max_epochs):
        epoch_start=time.perf_counter()
        data_wait=0.0; compute_time=0.0
        model.train(); train_loss_sum=train_correct=train_count=0
        batch_start=time.perf_counter()
        for images,labels in train_loader:
            data_wait += time.perf_counter()-batch_start
            compute_start=time.perf_counter()
            images=images.to(device,non_blocking=USE_AMP); labels=labels.to(device,non_blocking=USE_AMP)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP):
                outputs=model(images); loss=criterion(outputs,labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if USE_AMP: torch.cuda.synchronize()
            compute_time += time.perf_counter()-compute_start
            train_loss_sum += loss.item()*labels.size(0); train_correct += (outputs.argmax(dim=1)==labels).sum().item(); train_count += labels.size(0)
            batch_start=time.perf_counter()
        train_loss=train_loss_sum/train_count; train_accuracy=train_correct/train_count
        validation_start=time.perf_counter()
        model.eval(); val_loss_sum=val_correct=val_count=0
        with torch.no_grad():
            for images,labels in val_loader:
                images=images.to(device,non_blocking=USE_AMP); labels=labels.to(device,non_blocking=USE_AMP)
                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=USE_AMP):
                    outputs=model(images); loss=criterion(outputs,labels)
                val_loss_sum += loss.item()*labels.size(0); val_correct += (outputs.argmax(dim=1)==labels).sum().item(); val_count += labels.size(0)
        if USE_AMP: torch.cuda.synchronize()
        validation_time=time.perf_counter()-validation_start
        val_loss=val_loss_sum/val_count; val_accuracy=val_correct/val_count
        improved=val_loss < phase_best_val_loss; checkpoint_time=0.0
        if improved:
            phase_best_val_loss=val_loss; epochs_without_improvement=0
            if val_loss < best_val_loss:
                best_val_loss=val_loss
                best_model_weights={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
                save_start=time.perf_counter()
                torch.save({"model_state_dict":best_model_weights,"class_names":class_names,"class_to_idx":class_to_idx,"architecture":"mobilenet_v2","input_size":224,"best_val_loss":best_val_loss,"seed":SEED},CHECKPOINT_PATH)
                checkpoint_time=time.perf_counter()-save_start
        else: epochs_without_improvement += 1
        epoch_time=time.perf_counter()-epoch_start
        gpu_memory=torch.cuda.max_memory_allocated()/1024**3 if USE_AMP else 0.0
        history.append({"phase":phase_name,"epoch":epoch+1,"train_loss":train_loss,"train_accuracy":train_accuracy,"val_loss":val_loss,"val_accuracy":val_accuracy,"best":improved,"epochs_without_improvement":epochs_without_improvement,"data_wait_seconds":data_wait,"compute_seconds":compute_time,"validation_seconds":validation_time,"checkpoint_seconds":checkpoint_time,"epoch_seconds":epoch_time,"peak_gpu_memory_gb":gpu_memory})
        print(f"{phase_name} | Epoch {epoch+1}/{max_epochs} | train {train_loss:.4f}/{train_accuracy:.4f} | val {val_loss:.4f}/{val_accuracy:.4f} | data {data_wait:.1f}s | compute {compute_time:.1f}s | val {validation_time:.1f}s | save {checkpoint_time:.2f}s | total {epoch_time:.1f}s | VRAM {gpu_memory:.2f}GB | patience {epochs_without_improvement}/{PATIENCE}")
        if USE_AMP: torch.cuda.reset_peak_memory_stats()
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping {phase_name} after {epoch+1} epochs."); break

for parameter in model.parameters(): parameter.requires_grad=False
for parameter in model.classifier.parameters(): parameter.requires_grad=True
head_optimizer=optim.AdamW(model.classifier.parameters(),lr=1e-3,weight_decay=1e-4)
run_training_phase(head_optimizer,HEAD_EPOCHS,"Frozen backbone")
if best_model_weights is not None: model.load_state_dict(best_model_weights,strict=True)
for parameter in model.parameters(): parameter.requires_grad=True
finetune_optimizer=optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
run_training_phase(finetune_optimizer,FINETUNE_EPOCHS,"Fine-tuning")
if best_model_weights is not None: model.load_state_dict(best_model_weights,strict=True)
pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
print("Training finished.")
print("Best checkpoint:",CHECKPOINT_PATH.resolve())
print("Training history:",HISTORY_PATH.resolve())


C:\Users\rickv\AppData\Local\Temp\ipykernel_28008\3072767607.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)


Frozen backbone | Epoch 1/5 | train 0.7473/0.7381 | val 0.5467/0.8037 | data 21.4s | compute 7.4s | val 6.6s | save 0.01s | total 35.5s | VRAM 0.20GB | patience 0/5
Frozen backbone | Epoch 2/5 | train 0.5218/0.8129 | val 0.4797/0.8250 | data 18.1s | compute 8.7s | val 6.0s | save 0.01s | total 33.0s | VRAM 0.20GB | patience 0/5
Frozen backbone | Epoch 3/5 | train 0.4774/0.8219 | val 0.4286/0.8469 | data 17.8s | compute 8.6s | val 5.8s | save 0.01s | total 32.4s | VRAM 0.20GB | patience 0/5
Frozen backbone | Epoch 4/5 | train 0.4435/0.8372 | val 0.4126/0.8586 | data 17.6s | compute 8.6s | val 5.9s | save 0.01s | total 32.2s | VRAM 0.20GB | patience 0/5
Frozen backbone | Epoch 5/5 | train 0.4423/0.8360 | val 0.4148/0.8469 | data 17.8s | compute 8.6s | val 5.8s | save 0.00s | total 32.3s | VRAM 0.20GB | patience 1/5
Fine-tuning | Epoch 1/20 | train 0.3156/0.8835 | val 0.1762/0.9403 | data 20.0s | compute 10.4s | val 5.4s | save 0.01s | total 35.8s | VRAM 1.25GB | patience 0/5
Fine-tuning 